In [0]:
import numpy as np

from keras import (
    Sequential,
    regularizers,
)

from keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dropout,
    Dense,
)

from lib import (
    weight_to_rust,
    model_to_rust,
)

I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other 

# Load marvin word

In [0]:
# IMG_WIDTH, IMG_HEIGHT = 99, 43
# input = np.random.rand(1, IMG_WIDTH, IMG_HEIGHT, 1).astype(np.float32)
marvin = np.load("marvin.npz")["X"]
IMG_WIDTH, IMG_HEIGHT, CHANNEL = marvin.shape
# add batch dimension
marvin = marvin[np.newaxis, ...]
print(IMG_WIDTH, IMG_HEIGHT)

99 43


In [0]:
# https://github.com/atomic14/diy-alexa/blob/master/model/Train%20Model.ipynb
model = Sequential(
    [
        Conv2D(
            4,
            3,
            padding="same",
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001),
            name="conv_layer1",
            input_shape=(IMG_WIDTH, IMG_HEIGHT, 1),
        ),
        MaxPooling2D(name="max_pooling1", pool_size=(2, 2)),
        Conv2D(
            4,
            3,
            padding="same",
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001),
            name="conv_layer2",
        ),
        MaxPooling2D(name="max_pooling2", pool_size=(2, 2)),
        Flatten(),
        Dropout(0.2),
        Dense(
            40,
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001),
            name="hidden_layer1",
        ),
        Dense(
            1,
            activation="sigmoid",
            kernel_regularizer=regularizers.l2(0.001),
            name="output",
        ),
    ]
)
model.load_weights("./diy-alexa/model/trained.keras")

In [0]:
weights_rs = model_to_rust(model)
with open("../src/weights.rs", "w") as f:
    f.write(weights_rs)

In [0]:
output = model.predict(marvin)
marvin_test_data_rs = weight_to_rust("INPUT", marvin)
marvin_test_data_rs += weight_to_rust("OUTPUT", output)

with open("../tests/marvin_test_data.rs", "w") as f:
    f.write(marvin_test_data_rs)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
